In [ ]:
# !pip3 install --upgrade pip
# !pip3 install azure-ai-formrecognizer azure-core


In [1]:
import json
from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient

# ⚠️ Replace with your own endpoint & key
endpoint = "https://azuredocaiconversions.cognitiveservices.azure.com/"
key = "2ceebd0f9b5940bb9c7ac8524aa6d397"

# Local file path (e.g., PDF on your machine)
file_path = "/Users/aaditya/Documents/Projects/Reducto_steamlit/sample_for_reducto.pdf"

# Create client
document_analysis_client = DocumentAnalysisClient(
    endpoint=endpoint, credential=AzureKeyCredential(key)
)

# Open the local file and analyze
with open(file_path, "rb") as f:
    poller = document_analysis_client.begin_analyze_document(
        "prebuilt-document", document=f
    )
result = poller.result()

# Collect structured key-value pairs

# After collecting key-value pairs
kv_dict = {}
for kv_pair in result.key_value_pairs:
    key_text = kv_pair.key.content if kv_pair.key else None
    value_text = kv_pair.value.content if kv_pair.value else None
    if key_text:  # only add if key is not None
        kv_dict[key_text] = value_text

# JSON output as single dictionary
json_output = json.dumps(kv_dict, indent=4, ensure_ascii=False)
print(json_output)

print("------------------------------------------")


# Optional: save to file
with open("output.json", "w", encoding="utf-8") as f:
    f.write(json_output)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/aaditya/Documents/Projects/Reducto_steamlit/sample_for_reducto.pdf'

In [7]:
"""
Prebuilt US Personal Tax (Azure AI Document Intelligence) — Notebook version
"""

import os
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient


# ------------------------------ config ------------------------------ #
MODEL_ID = "prebuilt-tax.us"
FILEPATH = Path(r"/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.pdf")


# ------------------------------ helpers ------------------------------ #
def _val(field) -> Optional[str]:
    if field is None:
        return None
    for attr in ("value_string", "value_number", "value_address"):
        if hasattr(field, attr) and getattr(field, attr) is not None:
            return getattr(field, attr)
    return getattr(field, "content", None)


def _conf(field) -> Optional[float]:
    return getattr(field, "confidence", None) if field is not None else None


def _print_field(label: str, field) -> None:
    v, c = _val(field), _conf(field)
    if v is not None:
        print(f"{label}: {v} (conf: {c})")


# ------------------------------ client ------------------------------ #
def build_client() -> DocumentIntelligenceClient:
    load_dotenv()
    endpoint = os.getenv("AZURE_DOC_AI_ENDPOINT")
    key = os.getenv("AZURE_DOC_AI_KEY")

    if not endpoint or not key:
        raise RuntimeError(
            "Missing credentials. Set AZURE_DOC_AI_ENDPOINT and AZURE_DOC_AI_KEY in your .env or environment."
        )

    return DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))


# ------------------------------ analyze ------------------------------ #
def analyze_us_tax(filepath: Path, model_id: str = MODEL_ID):
    client = build_client()
    with filepath.open("rb") as f:
        poller = client.begin_analyze_document(model_id, body=f)
    return poller.result()


def pretty_print_us_tax_result(result) -> None:
    docs = getattr(result, "documents", []) or []
    if not docs:
        print("No documents recognized.")
        return

    for idx, document in enumerate(docs, start=1):
        print(f"\n-------- Recognizing document #{idx} --------")
        if getattr(document, "doc_type", None):
            print(f"Document Type: {document.doc_type}")

        fields = getattr(document, "fields", {}) or {}

        # Top-level
        _print_field("Tax Year", fields.get("TaxYear"))
        _print_field("W-2 Copy", fields.get("W2Copy"))
        _print_field("Wages, Tips, and Other Compensation", fields.get("WagesTipsAndOtherCompensation"))
        _print_field("Federal Income Tax Withheld", fields.get("FederalIncomeTaxWithheld"))
        _print_field("Social Security Wages", fields.get("SocialSecurityWages"))
        _print_field("Medicare Tax Withheld", fields.get("MedicareTaxWithheld"))

        # Employee
        emp = fields.get("Employee")
        if emp and getattr(emp, "value_object", None):
            v = emp.value_object
            print("Employee Information:")
            _print_field("  Name", v.get("Name"))
            _print_field("  SSN", v.get("SocialSecurityNumber"))
            _print_field("  Address", v.get("Address"))

        # Employer
        emp = fields.get("Employer")
        if emp and getattr(emp, "value_object", None):
            v = emp.value_object
            print("Employer Information:")
            _print_field("  Name", v.get("Name"))
            _print_field("  ID Number", v.get("IdNumber"))
            _print_field("  Address", v.get("Address"))

        # State taxes
        state_infos = fields.get("StateTaxInfos")
        if state_infos and getattr(state_infos, "value_array", None):
            print("State Tax Information:")
            for sidx, s in enumerate(state_infos.value_array, start=1):
                v = getattr(s, "value_object", {}) or {}
                st = v.get("State")
                if st:
                    print(f"  State #{sidx}: {_val(st)} (conf: {_conf(st)})")
                _print_field("    State Income Tax", v.get("StateIncomeTax"))

        # Additional info
        addl = fields.get("AdditionalInfo")
        if addl and getattr(addl, "value_array", None):
            print("Additional Information:")
            for aidx, info in enumerate(addl.value_array, start=1):
                v = getattr(info, "value_object", {}) or {}
                code, amt = v.get("LetterCode"), v.get("Amount")
                if code or amt:
                    print(
                        f"  Info #{aidx}: Code: {_val(code)}, Amount: {_val(amt)} "
                        f"(conf: {_conf(code) or _conf(amt)})"
                    )

        print("--------------------------------------")


# ------------------------------ run in notebook ------------------------------ #
result = analyze_us_tax(FILEPATH)
pretty_print_us_tax_result(result)



-------- Recognizing document #1 --------
Document Type: tax.us.1099B.2022
Tax Year: 2024 (conf: 0.622)
--------------------------------------


In [8]:
"""
Prebuilt US Personal Tax (Azure AI Document Intelligence) — Notebook version with 1099-B support & field dump
"""

import os
from pathlib import Path
from typing import Optional, Any, Dict

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

# ------------------------------ config ------------------------------ #
MODEL_ID = "prebuilt-tax.us"
FILEPATH = Path(r"/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.pdf")

# ------------------------------ helpers ------------------------------ #
def _val(field) -> Optional[Any]:
    if field is None:
        return None
    for attr in ("value_string", "value_number", "value_address", "value_currency", "value_date"):
        if hasattr(field, attr) and getattr(field, attr) is not None:
            return getattr(field, attr)
    return getattr(field, "content", None)

def _conf(field) -> Optional[float]:
    return getattr(field, "confidence", None) if field is not None else None

def _print_field(label: str, field) -> None:
    v, c = _val(field), _conf(field)
    if v is not None:
        print(f"{label}: {v} (conf: {c})")

def _dump_fields(fields: Dict[str, Any], prefix: str = "") -> None:
    """
    Generic, form-agnostic dump of everything Azure returned so you can discover keys.
    Handles nested objects and arrays.
    """
    for k, f in (fields or {}).items():
        if f is None:
            continue
        if hasattr(f, "value_object") and f.value_object:
            print(f"{prefix}{k}: <object> (conf: {_conf(f)})")
            _dump_fields(f.value_object, prefix + "  ")
        elif hasattr(f, "value_array") and f.value_array:
            print(f"{prefix}{k}: <array> (len={len(f.value_array)}, conf: {_conf(f)})")
            for i, item in enumerate(f.value_array, start=1):
                if hasattr(item, "value_object") and item.value_object:
                    print(f"{prefix}  [{i}] <object>")
                    _dump_fields(item.value_object, prefix + "    ")
                else:
                    print(f"{prefix}  [{i}] {_val(item)} (conf: {_conf(item)})")
        else:
            print(f"{prefix}{k}: {_val(f)} (conf: {_conf(f)})")

# ------------------------------ client ------------------------------ #
def build_client() -> DocumentIntelligenceClient:
    load_dotenv()
    endpoint = os.getenv("AZURE_DOC_AI_ENDPOINT")
    key = os.getenv("AZURE_DOC_AI_KEY")
    if not endpoint or not key:
        raise RuntimeError("Set AZURE_DOC_AI_ENDPOINT and AZURE_DOC_AI_KEY in your env/.env")
    return DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

# ------------------------------ analyze ------------------------------ #
def analyze_us_tax(filepath: Path, model_id: str = MODEL_ID):
    client = build_client()
    with filepath.open("rb") as f:
        poller = client.begin_analyze_document(model_id, body=f)
    return poller.result()

def pretty_print_generic(fields) -> None:
    # Generic fields commonly present across forms
    _print_field("Tax Year", fields.get("TaxYear"))
    _print_field("Federal Income Tax Withheld", fields.get("FederalIncomeTaxWithheld"))
    _print_field("Medicare Tax Withheld", fields.get("MedicareTaxWithheld"))
    _print_field("Social Security Wages", fields.get("SocialSecurityWages"))

    # Parties
    for label in ("Employee", "Recipient", "Payer", "Employer"):
        node = fields.get(label)
        if node and getattr(node, "value_object", None):
            v = node.value_object
            print(f"{label} Information:")
            _print_field("  Name", v.get("Name"))
            _print_field("  SSN/TIN", v.get("SocialSecurityNumber") or v.get("TaxpayerIdentificationNumber") or v.get("TIN"))
            _print_field("  Address", v.get("Address"))
            _print_field("  Id Number", v.get("IdNumber"))

def pretty_print_1099b(fields) -> None:
    """
    Heuristic 1099-B printer; prints only if keys exist.
    Names can vary by SDK version/model; the field dump below reveals exact keys.
    """
    print("— 1099-B fields (if present) —")
    likely_keys = {
        "Proceeds": "Proceeds",
        "CostOrOtherBasis": "Cost or Other Basis",
        "WashSaleLossDisallowed": "Wash Sale Loss Disallowed",
        "FederalIncomeTaxWithheld": "Federal Income Tax Withheld",
        "CUSIPNumber": "CUSIP",
        "Description": "Description",
        "DateOfAcquisition": "Date Acquired",
        "DateOfSale": "Date Sold",
        "AccruedMarketDiscount": "Accrued Market Discount",
        "Adjustments": "Adjustments",
        # Parties
        "Payer": "Payer",
        "Recipient": "Recipient",
        "AccountNumber": "Account Number",
    }
    for key, label in likely_keys.items():
        f = fields.get(key)
        if f is None:
            continue
        if hasattr(f, "value_object") and f.value_object:
            print(f"{label}:")
            _dump_fields(f.value_object, prefix="  ")
        else:
            _print_field(label, f)

def pretty_print_us_tax_result(result) -> None:
    docs = getattr(result, "documents", []) or []
    if not docs:
        print("No documents recognized.")
        return

    for idx, document in enumerate(docs, start=1):
        print(f"\n-------- Recognizing document #{idx} --------")
        doc_type = getattr(document, "doc_type", None)
        if doc_type:
            print(f"Document Type: {doc_type}")

        fields = getattr(document, "fields", {}) or {}

        # First: dump everything so you see exact keys Azure returned
        print("\nAll fields (raw dump):")
        _dump_fields(fields, prefix="  ")

        # Then: targeted pretty-prints
        print("\nSelected fields:")
        pretty_print_generic(fields)
        if doc_type and doc_type.startswith("tax.us.1099B"):
            pretty_print_1099b(fields)

        print("--------------------------------------")

# ------------------------------ run in notebook ------------------------------ #
result = analyze_us_tax(FILEPATH)
pretty_print_us_tax_result(result)



-------- Recognizing document #1 --------
Document Type: tax.us.1099B.2022

All fields (raw dump):
  TaxYear: 2024 (conf: 0.622)
  Payer: <object> (conf: None)
    TIN: 123654987 (conf: 0.79)
    Name: Fname GenInfo (conf: 0.715)
    Address: {'city': 'Mountain', 'streetAddress': ''} (conf: 0.465)
    PhoneNumber: None (conf: 0.84)
  Recipient: <object> (conf: None)
    TIN: None (conf: 0.443)
    Name: Fname
GenInfo (conf: 0.418)
    Address: {'houseNumber': '444', 'road': 'mainstreet\napt24', 'city': 'Iron Mountain', 'state': 'Michigan', 'streetAddress': '444 mainstreet\napt24'} (conf: 0.262)
    AccountNumber: None (conf: 0.481)
  Transactions: <array> (len=1, conf: None)
    [1] <object>
      CusipNumber: 123-65-4987 (conf: 0.55)
      IsFactaFilingRequired: None (conf: 0.711)
      ApplicableForm8949Checkbox: None (conf: 0.984)
      Box1a: None (conf: 0.613)
      Box1b: None (conf: 0.612)
      Box1c: None (conf: 0.561)
      Box1d: None (conf: 0.605)
      Box1e: None (conf: 

In [9]:
"""
Azure Document Intelligence — Prebuilt US Tax (max info, notebook version)
- Field dump (recursive)
- Key-Value pairs & Tables extraction
- Flatten to dict for DataFrame
- Save raw JSON for debugging
"""

import os, json
from pathlib import Path
from typing import Optional, Any, Dict, List, Tuple

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

# Optional: DataFrame view
try:
    import pandas as pd
except Exception:
    pd = None

# ------------------------------ config ------------------------------ #
MODEL_ID = "prebuilt-tax.us"
FILEPATH = Path(r"/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.pdf")
OUT_JSON = FILEPATH.with_suffix(".docintelligence.json")

# ------------------------------ helpers ------------------------------ #
def _val(field) -> Optional[Any]:
    if field is None:
        return None
    for attr in ("value_string", "value_number", "value_address", "value_currency", "value_date", "value_phone_number", "value_time"):
        if hasattr(field, attr) and getattr(field, attr) is not None:
            return getattr(field, attr)
    return getattr(field, "content", None)

def _conf(field) -> Optional[float]:
    return getattr(field, "confidence", None) if field is not None else None

def _dump_fields(fields: Dict[str, Any], prefix: str = "") -> None:
    for k, f in (fields or {}).items():
        if f is None:
            continue
        if hasattr(f, "value_object") and f.value_object:
            print(f"{prefix}{k}: <object> (conf: {_conf(f)})")
            _dump_fields(f.value_object, prefix + "  ")
        elif hasattr(f, "value_array") and f.value_array:
            print(f"{prefix}{k}: <array> (len={len(f.value_array)}, conf: {_conf(f)})")
            for i, item in enumerate(f.value_array, start=1):
                if hasattr(item, "value_object") and item.value_object:
                    print(f"{prefix}  [{i}] <object>")
                    _dump_fields(item.value_object, prefix + "    ")
                else:
                    print(f"{prefix}  [{i}] {_val(item)} (conf: {_conf(item)})")
        else:
            print(f"{prefix}{k}: {_val(f)} (conf: {_conf(f)})")

def _flatten_fields(fields: Dict[str, Any], prefix: str = "", out: Dict[str, Any] = None) -> Dict[str, Any]:
    """
    Flattens all fields (objects/arrays) into a single dict with dotted keys.
    Example key: Employee.Name, StateTaxInfos[0].State
    """
    if out is None:
        out = {}
    for k, f in (fields or {}).items():
        key = f"{prefix}{k}" if not prefix else f"{prefix}.{k}"
        if f is None:
            out[key] = None
            continue
        if hasattr(f, "value_object") and f.value_object:
            _flatten_fields(f.value_object, key, out)
        elif hasattr(f, "value_array") and f.value_array:
            for i, item in enumerate(f.value_array):
                idx_key = f"{key}[{i}]"
                if hasattr(item, "value_object") and item.value_object:
                    _flatten_fields(item.value_object, idx_key, out)
                else:
                    out[idx_key] = _val(item)
                    out[idx_key + ".__conf"] = _conf(item)
        else:
            out[key] = _val(f)
            out[key + ".__conf"] = _conf(f)
    return out

def _extract_kvps(result) -> List[Tuple[str, str, float]]:
    """
    Extract generic key-value pairs if the service returned them.
    (Available in v4 AnalyzeResult as key_value_pairs; if absent, this will return an empty list.)
    """
    kvps = []
    kv_list = getattr(result, "key_value_pairs", None)
    if not kv_list:
        return kvps
    for kv in kv_list:
        k = getattr(kv, "key", None)
        v = getattr(kv, "value", None)
        conf = getattr(kv, "confidence", None)
        k_text = getattr(k, "content", None) if k is not None else None
        v_text = getattr(v, "content", None) if v is not None else None
        kvps.append((k_text, v_text, conf))
    return kvps

def _extract_tables(result) -> List[List[List[str]]]:
    """
    Returns tables as a list of tables; each table is a list of rows; each row is a list of cell texts.
    """
    out = []
    tables = getattr(result, "tables", None)
    if not tables:
        return out
    for t in tables:
        # Collect by (row, col)
        grid = {}
        max_r = max_c = -1
        for cell in t.cells:
            r, c = cell.row_index, cell.column_index
            txt = getattr(cell, "content", "") or ""
            grid[(r, c)] = txt
            max_r, max_c = max(max_r, r), max(max_c, c)
        rows = []
        for r in range(max_r + 1):
            row = []
            for c in range(max_c + 1):
                row.append(grid.get((r, c), ""))
            rows.append(row)
        out.append(rows)
    return out

# ------------------------------ client ------------------------------ #
def build_client() -> DocumentIntelligenceClient:
    load_dotenv()
    endpoint = os.getenv("AZURE_DOC_AI_ENDPOINT")
    key = os.getenv("AZURE_DOC_AI_KEY")
    if not endpoint or not key:
        raise RuntimeError("Set AZURE_DOC_AI_ENDPOINT and AZURE_DOC_AI_KEY in your env/.env")
    return DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

# ------------------------------ analyze ------------------------------ #
def analyze_us_tax(filepath: Path, model_id: str = MODEL_ID):
    client = build_client()
    with filepath.open("rb") as f:
        poller = client.begin_analyze_document(model_id, body=f)
    return poller.result()

# ------------------------------ run & display ------------------------------ #
result = analyze_us_tax(FILEPATH)

docs = getattr(result, "documents", []) or []
print(f"Documents detected: {len(docs)}")
for i, d in enumerate(docs, 1):
    print(f"\n=== Document #{i} ===")
    print("doc_type:", getattr(d, "doc_type", None))
    fields = getattr(d, "fields", {}) or {}

    # 1) Full field dump for discovery
    print("\n-- Field dump --")
    _dump_fields(fields, prefix="  ")

    # 2) Flatten for DataFrame/dicts
    flat = _flatten_fields(fields)
    print("\n-- Flattened keys (sample) --")
    for j, (k, v) in enumerate(flat.items()):
        if j >= 20:  # don’t flood output
            print("  ...")
            break
        print(f"  {k}: {v}")

    # 3) KVPs
    kv_pairs = _extract_kvps(result)
    if kv_pairs:
        print("\n-- Key-Value Pairs --")
        for k, v, c in kv_pairs[:20]:
            print(f"  {k} -> {v} (conf: {c})")
        if len(kv_pairs) > 20:
            print("  ...")

    # 4) Tables
    tables = _extract_tables(result)
    if tables:
        print(f"\n-- Tables: {len(tables)} found --")
        for t_idx, table in enumerate(tables, 1):
            print(f"  Table {t_idx}: {len(table)} rows x {len(table[0]) if table else 0} cols")
            for row in table[:5]:
                print("   ", row)
            if len(table) > 5:
                print("    ...")

    # 5) Optional DataFrame row
    if pd is not None:
        df = pd.DataFrame([flat])
        display(df)

# 6) Save raw JSON snapshot (best for deep debugging)
try:
    # result.as_dict() exists on v4 models; if not, fall back to manual JSON via attributes.
    if hasattr(result, "as_dict"):
        raw = result.as_dict()
    else:
        # Minimal fallback; not exhaustive.
        raw = {
            "content": getattr(result, "content", None),
            "documents": [{
                "doc_type": getattr(d, "doc_type", None),
            } for d in docs],
        }
    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)
    print(f"\nSaved raw JSON → {OUT_JSON}")
except Exception as e:
    print("JSON save skipped:", e)


Documents detected: 1

=== Document #1 ===
doc_type: tax.us.1099B.2022

-- Field dump --
  TaxYear: 2024 (conf: 0.622)
  Payer: <object> (conf: None)
    TIN: 123654987 (conf: 0.79)
    Name: Fname GenInfo (conf: 0.715)
    Address: {'city': 'Mountain', 'streetAddress': ''} (conf: 0.465)
    PhoneNumber: None (conf: 0.84)
  Recipient: <object> (conf: None)
    TIN: None (conf: 0.443)
    Name: Fname
GenInfo (conf: 0.418)
    Address: {'houseNumber': '444', 'road': 'mainstreet\napt24', 'city': 'Iron Mountain', 'state': 'Michigan', 'streetAddress': '444 mainstreet\napt24'} (conf: 0.262)
    AccountNumber: None (conf: 0.481)
  Transactions: <array> (len=1, conf: None)
    [1] <object>
      CusipNumber: 123-65-4987 (conf: 0.55)
      IsFactaFilingRequired: None (conf: 0.711)
      ApplicableForm8949Checkbox: None (conf: 0.984)
      Box1a: None (conf: 0.613)
      Box1b: None (conf: 0.612)
      Box1c: None (conf: 0.561)
      Box1d: None (conf: 0.605)
      Box1e: None (conf: 0.634)
    

,TaxYear,TaxYear.__conf,Payer.TIN,Payer.TIN.__conf,Payer.Name,Payer.Name.__conf,Payer.Address,Payer.Address.__conf,Payer.PhoneNumber,Payer.PhoneNumber.__conf,...,Transactions[0].StateTaxesWithheld[0].Box16,Transactions[0].StateTaxesWithheld[0].Box16.__conf,Transactions[0].StateTaxesWithheld[1].Box14,Transactions[0].StateTaxesWithheld[1].Box14.__conf,Transactions[0].StateTaxesWithheld[1].Box15,Transactions[0].StateTaxesWithheld[1].Box15.__conf,Transactions[0].StateTaxesWithheld[1].Box16,Transactions[0].StateTaxesWithheld[1].Box16.__conf,IsCorrected,IsCorrected.__conf
0,2024,0.622,123654987,0.79,Fname GenInfo,0.715,"[city, streetAddress]",0.465,None,0.84,...,None,0.94,None,0.823,None,0.822,None,0.97,None,0.984



Saved raw JSON → /Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.docintelligence.json


In [10]:
# Azure Doc Intelligence — auto classify + map to schema
import os, re
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient

MODEL_ID = "prebuilt-tax.us"
DOC_CONF_MIN = 0.55   # doc-type confidence gate
FIELD_CONF_MIN = 0.50 # field gate

# ---------- helpers ----------
def _val(f):
    if f is None: return None
    for attr in ("value_string","value_number","value_address","value_currency","value_date","value_phone_number","value_time"):
        v = getattr(f, attr, None)
        if v is not None: return v
    return getattr(f, "content", None)

def _conf(f) -> Optional[float]:
    return getattr(f, "confidence", None) if f is not None else None

def _keep(f, thr=FIELD_CONF_MIN) -> bool:
    return f is not None and (_conf(f) is None or _conf(f) >= thr)

def _addr_one_line(a: Any) -> Optional[str]:
    if not isinstance(a, dict): return (str(a).strip() or None)
    parts = [a.get(k) for k in ("streetAddress","houseNumber","road","city","state","postalCode","countryRegion")]
    parts = [str(x).replace("\n"," ").strip() for x in parts if x]
    seen, out = set(), []
    for p in parts:
        if p not in seen:
            out.append(p); seen.add(p)
    return ", ".join(out) or None

def _clean_name(f): 
    v = _val(f); 
    return re.sub(r"\s+"," ",str(v)).strip() if v else None

def _clean_tin(f):
    v = _val(f)
    if v is None: return None
    return re.sub(r"[^\d-]","",str(v))

def _clean_cusip(f):
    v = _val(f)
    return str(v).replace(" ","").upper() if v else None

def _normalize_doc_type(dt: Optional[str]) -> Optional[str]:
    # e.g., tax.us.1099B.2022 -> tax.us.1099B  (keep prefix for routing)
    if not dt: return None
    m = re.match(r"^(tax\.us\.[^.]+)", dt)
    return m.group(1) if m else dt

# ---------- mappers ----------
def map_1099b(fields: Dict[str, Any]) -> Dict[str, Any]:
    payer = fields.get("Payer").value_object if fields.get("Payer") and getattr(fields.get("Payer"),"value_object",None) else {}
    recip = fields.get("Recipient").value_object if fields.get("Recipient") and getattr(fields.get("Recipient"),"value_object",None) else {}
    txs_raw = fields.get("Transactions")

    header = {
        "tax_year": _val(fields.get("TaxYear")) if _keep(fields.get("TaxYear")) else None,
        "is_corrected": _val(fields.get("IsCorrected")) if _keep(fields.get("IsCorrected")) else None,
        "payer_name": _clean_name(payer.get("Name")) if _keep(payer.get("Name")) else None,
        "payer_tin": _clean_tin(payer.get("TIN")) if _keep(payer.get("TIN")) else None,
        "payer_phone": _val(payer.get("PhoneNumber")) if _keep(payer.get("PhoneNumber")) else None,
        "payer_address": _addr_one_line(_val(payer.get("Address"))) if _keep(payer.get("Address")) else None,
        "recipient_name": _clean_name(recip.get("Name")) if _keep(recip.get("Name")) else None,
        "recipient_tin": _clean_tin(recip.get("TIN")) if _keep(recip.get("TIN")) else None,
        "recipient_account_number": _val(recip.get("AccountNumber")) if _keep(recip.get("AccountNumber")) else None,
        "recipient_address": _addr_one_line(_val(recip.get("Address"))) if _keep(recip.get("Address")) else None,
    }

    transactions: List[Dict[str, Any]] = []
    if txs_raw and getattr(txs_raw, "value_array", None):
        for item in txs_raw.value_array:
            v = getattr(item, "value_object", {}) or {}
            row = {
                "cusip": _clean_cusip(v.get("CusipNumber")) if _keep(v.get("CusipNumber")) else None,
                "box1a": _val(v.get("Box1a")) if _keep(v.get("Box1a")) else None,   # description
                "box1b": _val(v.get("Box1b")) if _keep(v.get("Box1b")) else None,   # date acquired
                "box1c": _val(v.get("Box1c")) if _keep(v.get("Box1c")) else None,   # date sold
                "box1d": _val(v.get("Box1d")) if _keep(v.get("Box1d")) else None,   # proceeds
                "box1e": _val(v.get("Box1e")) if _keep(v.get("Box1e")) else None,   # cost/basis
                "box1f": _val(v.get("Box1f")) if _keep(v.get("Box1f")) else None,   # wash sale
                "box1g": _val(v.get("Box1g")) if _keep(v.get("Box1g")) else None,   # adjustments
                "box4": _val(v.get("Box4")) if _keep(v.get("Box4")) else None,      # federal tax withheld
            }
            # state taxes sub-array
            st = v.get("StateTaxesWithheld")
            states = []
            if st and getattr(st, "value_array", None):
                for s in st.value_array:
                    o = getattr(s, "value_object", {}) or {}
                    states.append({
                        "box14": _val(o.get("Box14")) if _keep(o.get("Box14")) else None,
                        "box15": _val(o.get("Box15")) if _keep(o.get("Box15")) else None,
                        "box16": _val(o.get("Box16")) if _keep(o.get("Box16")) else None,
                    })
            row["state_taxes_withheld"] = states or None
            transactions.append(row)

    return {"header": header, "transactions": transactions}

def map_w2(fields: Dict[str, Any]) -> Dict[str, Any]:
    # Minimal example; flesh out as needed
    emp = fields.get("Employee").value_object if fields.get("Employee") and getattr(fields.get("Employee"),"value_object",None) else {}
    empl = fields.get("Employer").value_object if fields.get("Employer") and getattr(fields.get("Employer"),"value_object",None) else {}
    return {
        "header": {
            "tax_year": _val(fields.get("TaxYear")) if _keep(fields.get("TaxYear")) else None,
            "employee_name": _clean_name(emp.get("Name")) if _keep(emp.get("Name")) else None,
            "employee_ssn": _clean_tin(emp.get("SocialSecurityNumber")) if _keep(emp.get("SocialSecurityNumber")) else None,
            "employer_name": _clean_name(empl.get("Name")) if _keep(empl.get("Name")) else None,
            "employer_id": _clean_tin(empl.get("IdNumber")) if _keep(empl.get("IdNumber")) else None,
        }
    }

MAPPERS = {
    "tax.us.1099B": map_1099b,
    "tax.us.w2": map_w2,
    # add: "tax.us.1099DIV": map_1099div, "tax.us.1099INT": map_1099int, "tax.us.1040": map_1040, ...
}

# ---------- client + pipeline ----------
def _client() -> DocumentIntelligenceClient:
    load_dotenv()
    ep, key = os.getenv("AZURE_DOC_AI_ENDPOINT"), os.getenv("AZURE_DOC_AI_KEY")
    if not ep or not key:
        raise RuntimeError("Set AZURE_DOC_AI_ENDPOINT and AZURE_DOC_AI_KEY")
    return DocumentIntelligenceClient(endpoint=ep, credential=AzureKeyCredential(key))

def analyze_and_route(path: Path) -> Dict[str, Any]:
    client = _client()
    with path.open("rb") as f:
        poller = client.begin_analyze_document(MODEL_ID, body=f)
    result = poller.result()

    docs = getattr(result, "documents", []) or []
    if not docs:
        return {"error": "No documents recognized"}

    doc = docs[0]
    doc_type_full = getattr(doc, "doc_type", None)
    doc_conf = getattr(doc, "confidence", None)
    doc_type_base = _normalize_doc_type(doc_type_full)

    routed = {"doc_type": doc_type_full, "doc_type_base": doc_type_base, "doc_confidence": doc_conf}

    fields = getattr(doc, "fields", {}) or {}

    # gate by doc confidence; if low, you can fall back to a custom classifier (see below)
    if doc_conf is not None and doc_conf < DOC_CONF_MIN:
        routed["warning"] = f"Low doc_type confidence ({doc_conf:.2f}); consider custom classifier routing."
    mapper = MAPPERS.get(doc_type_base)
    if mapper:
        routed["mapped"] = mapper(fields)
    else:
        routed["mapped"] = {"header": {"tax_year": _val(fields.get("TaxYear")) if _keep(fields.get("TaxYear")) else None}}
        routed["warning"] = (routed.get("warning") or "") + " Unhandled doc_type; using minimal mapping."
    return routed

# ---- run it on your file ----
data = analyze_and_route(Path(r"/Users/aaditya/Documents/Projects/Reducto_steamlit/testing_files/sample_for_reducto.pdf"))
data


{'doc_type': 'tax.us.1099B.2022',
 'doc_type_base': 'tax.us.1099B',
 'doc_confidence': 0.263,
 'warning': 'Low doc_type confidence (0.26); consider custom classifier routing.',
 'mapped': {'header': {'tax_year': '2024',
   'is_corrected': None,
   'payer_name': 'Fname GenInfo',
   'payer_tin': '123654987',
   'payer_phone': None,
   'payer_address': None,
   'recipient_name': None,
   'recipient_tin': None,
   'recipient_account_number': None,
   'recipient_address': None},
  'transactions': [{'cusip': '123-65-4987',
    'box1a': None,
    'box1b': None,
    'box1c': None,
    'box1d': None,
    'box1e': None,
    'box1f': None,
    'box1g': None,
    'box4': None,
    'state_taxes_withheld': [{'box14': None, 'box15': None, 'box16': None},
     {'box14': None, 'box15': None, 'box16': None}]}]}}